In [1]:
import os
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datasets import load_dataset

/usr/workspace/wsb/kirchenb/tuolumne_conda_28_630_fiction/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BASE_OUTPUT_PATH = "/p/vast1/kirchenb/fiction_stash/lit-gpt-dev-fiction/output"

# "/p/vast1/kirchenb/fiction_stash/lit-gpt-dev-fiction/output/v6_exp1_train_val_splits_5pct_seed1234/exp1_train_val_splits_5pct_4N_mb8-wb128_llama-3-1-8B_event-split-fictions-train-val/lm_eval_results/hf_checkpoint_step-00000499-exp1_train_val_splits_5pct_4N_mb8-wb128_llama-3-1-8B_event-split-fictions-train-val/samples_fict_qa_obqa_blind_inf_ex_dedup_ds_mcq_2025-04-07T11-10-31.722834.jsonl"

# full performance breakdown is in
# "/p/vast1/kirchenb/fiction_stash/lit-gpt-dev-fiction/output/v6_exp1_train_val_splits_5pct_seed1234/exp1_train_val_splits_5pct_4N_mb8-wb128_llama-3-1-8B_event-split-fictions-train-val/lm_eval_results/hf_checkpoint_step-00000499-exp1_train_val_splits_5pct_4N_mb8-wb128_llama-3-1-8B_event-split-fictions-train-val/results_2025-04-13T13-47-48.383939.json"

RUN_GRP = "v6_exp1_train_val_splits_5pct_seed1234"
RUN_NAME = "exp1_train_val_splits_5pct_4N_mb8-wb128_llama-3-1-8B_event-split-fictions-train-val"
RESULT_SUBDIR = "lm_eval_results"
INIT_CKPT_STEP = "hf_checkpoint_step-00000000"
FINAL_CKPT_STEP = "hf_checkpoint_step-00000499"
INIT_FILENAME = "samples_fict_qa_obqa_blind_inf_ex_dedup_ds_mcq_2025-04-07T10-34-04.100979.jsonl"
FINAL_FILENAME = "samples_fict_qa_obqa_blind_inf_ex_dedup_ds_mcq_2025-04-07T11-10-31.722834.jsonl"

INIT_FULL_JSONL_PATH = f"{BASE_OUTPUT_PATH}/{RUN_GRP}/{RUN_NAME}/{RESULT_SUBDIR}/{INIT_CKPT_STEP}-{RUN_NAME}/{INIT_FILENAME}"
FINAL_FULL_JSONL_PATH = f"{BASE_OUTPUT_PATH}/{RUN_GRP}/{RUN_NAME}/{RESULT_SUBDIR}/{FINAL_CKPT_STEP}-{RUN_NAME}/{FINAL_FILENAME}"

In [3]:
def load_jsonl_to_dataframe(file_path: str) -> pd.DataFrame:
    """
    Loads data from a JSON Lines (.jsonl) file into a flattened pandas DataFrame.

    This function reads the file line by line, parses each line as a JSON object,
    and uses pd.json_normalize to automatically flatten nested dictionaries, 
    creating clear column names (e.g., 'doc_event_id', 'doc_natural_answer').

    Args:
        file_path: The full path to the .jsonl file.

    Returns:
        A pandas DataFrame containing the flattened tabular data.
    """
    try:
        # 1. Read the JSONL file line by line and parse it into a list of dictionaries
        with open(file_path, 'r', encoding='utf-8') as f:
            data = [json.loads(line) for line in f]

        # 2. Use pd.json_normalize to flatten the data
        # 'sep' defines the separator for nested keys (e.g., 'doc' + '_' + 'event_id')
        df = pd.json_normalize(
            data,
            sep='_',
            errors='ignore' # Use 'ignore' to skip lines or fields that don't conform
        )
        
        print(f"Successfully loaded and flattened {len(df)} records.")
        return df

    except FileNotFoundError:
        print(f"Error: File not found at {file_path}")
        return pd.DataFrame()
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}")
        return pd.DataFrame()
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return pd.DataFrame()

def extract_choice_from_jsonl_df(df):
    """ Model attempt is specified by a series of scores for each choice under the "resps" key.
    
    "resps": [[["-19.20279312133789", "False"]], [["-27.694900512695312", "False"]], [["-26.183048248291016", "False"]], [["-31.78978729248047", "False"]]]
    """
    choice_indices = []
    ranking_lists = []
    for idx, row in df.iterrows():
        resps = row['resps']
        # extract the scores for each choice
        scores = [float(choice[0][0]) for choice in resps]
        # find the index of the choice with the highest score (least negative)
        best_choice_index = int(np.argmax(scores))
        ranking = np.argsort(scores)[::-1]  # descending order
        choice_indices.append(best_choice_index)
        ranking_lists.append(ranking.tolist())
    df['model_choice'] = choice_indices
    df['choice_ranking'] = ranking_lists

    return df

def basic_performance_analysis(df: pd.DataFrame, verdict_col_name: str = "acc") -> None:
    """
    Performs basic performance analysis on the MCQ results DataFrame.

    This function calculates and prints the overall accuracy, 
    as well as accuracy per question type if available.

    Args:
        df: The pandas DataFrame containing the MCQ results.
    """
    if df.empty:
        print("DataFrame is empty. Cannot perform analysis.")
        return

    # Calculate overall accuracy
    total_questions = len(df)
    verdict_col_data = df[verdict_col_name].astype(bool)
    correct_answers = df[verdict_col_name].sum()
    overall_accuracy = correct_answers / total_questions if total_questions > 0 else 0

    print(f"Overall Accuracy: {overall_accuracy:.2%} ({correct_answers}/{total_questions})")

    # If 'question_type' column exists, calculate accuracy per question type
    if 'question_type' in df.columns:
        accuracy_by_type = df.groupby('question_type')['is_correct'].mean()
        print("\nAccuracy by Question Type:")
        for q_type, acc in accuracy_by_type.items():
            print(f"  {q_type}: {acc:.2%}")

def pull_raw_dataset(hf_repo_id: str, config_name: str, split:str="train"):
    dataset = load_dataset(hf_repo_id, config_name, split=split)
    # convert to dataframe
    df = dataset.to_pandas()
    return df

In [4]:
df_lm_eval_init_result = load_jsonl_to_dataframe(INIT_FULL_JSONL_PATH)
df_lm_eval_final_result = load_jsonl_to_dataframe(FINAL_FULL_JSONL_PATH)
# df_lm_eval_final_result

df_lm_eval_init_result = extract_choice_from_jsonl_df(df_lm_eval_init_result)
df_lm_eval_final_result = extract_choice_from_jsonl_df(df_lm_eval_final_result)


df_lm_eval_final_result["acc_init"] = df_lm_eval_init_result["acc"]
df_lm_eval_final_result["acc_final"] = df_lm_eval_final_result["acc"]
df_lm_eval_final_result["model_choice_init"] = df_lm_eval_init_result["model_choice"]
df_lm_eval_final_result["model_choice_final"] = df_lm_eval_final_result["model_choice"]
df_lm_eval_final_result["choice_ranking_init"] = df_lm_eval_init_result["choice_ranking"]
df_lm_eval_final_result["choice_ranking_final"] = df_lm_eval_final_result["choice_ranking"]

df_lm_eval_final_result['target'] = df_lm_eval_final_result['target'].astype(int)

# there seems to be a spurious answer "Western and Eastern"
# make a col indicating when it is in the choice list
df_lm_eval_final_result['W_and_E_in_choices'] = df_lm_eval_final_result['doc_topk_choices'].apply(lambda choices: "Western and Eastern" in choices)
df_lm_eval_final_result['W_and_E_is_model_choice_init'] = df_lm_eval_final_result[['model_choice_init', 'doc_topk_choices']].apply(
    lambda row: row['doc_topk_choices'][row['model_choice_init']] == "Western and Eastern", axis=1
) 
df_lm_eval_final_result['W_and_E_is_model_choice_final'] = df_lm_eval_final_result[['model_choice_final', 'doc_topk_choices']].apply(
    lambda row: row['doc_topk_choices'][row['model_choice_final']] == "Western and Eastern", axis=1
)

df_lm_eval_final_result

Successfully loaded and flattened 3036 records.
Successfully loaded and flattened 3036 records.


,doc_id,target,resps,filtered_resps,filter,metrics,doc_hash,prompt_hash,target_hash,acc,...,choice_ranking,acc_init,acc_final,model_choice_init,model_choice_final,choice_ranking_init,choice_ranking_final,W_and_E_in_choices,W_and_E_is_model_choice_init,W_and_E_is_model_choice_final
0,0,2,"[[[-46.50212860107422, False]], [[-38.50323104...","[[-46.50212860107422, False], [-38.50323104858...",none,"[acc, acc_norm]",c729dfa8379ae2dbcb1c7f65117aac63bbd716e3131164...,f3cff83b03a08fc897cb5b798abf6f616e202574fce38e...,d4735e3a265e16eee03f59718b9b5d03019c07d8b6c51f...,1.0,...,"[2, 3, 1, 0]",1.0,1.0,2,2,"[2, 3, 1, 0]","[2, 3, 1, 0]",False,False,False
1,1,0,"[[[-24.992748260498047, False]], [[-24.4700202...","[[-24.992748260498047, False], [-24.4700202941...",none,"[acc, acc_norm]",4d44cd3433eb55ce4b9bd76fb12dca098a8cc2ae21bf2a...,e45a87c16ce89047fc3377f0d7f8dd136e06f68a24e57b...,5feceb66ffc86f38d952786c6d696c79c2dbc239dd4e91...,0.0,...,"[3, 1, 0, 2]",0.0,0.0,3,3,"[3, 1, 0, 2]","[3, 1, 0, 2]",False,False,False
2,2,2,"[[[-17.62920379638672, False]], [[-27.24129104...","[[-17.62920379638672, False], [-27.24129104614...",none,"[acc, acc_norm]",31f9277df4adea7a08fc8914c8f01d181c438c7c6b69f7...,16a6d8f1c33f66cdfb3b85d888205d6eb98410e852eaea...,d4735e3a265e16eee03f59718b9b5d03019c07d8b6c51f...,0.0,...,"[0, 2, 3, 1]",0.0,0.0,0,0,"[0, 2, 1, 3]","[0, 2, 3, 1]",False,False,False
3,3,2,"[[[-33.734130859375, False]], [[-20.2086391448...","[[-33.734130859375, False], [-20.2086391448974...",none,"[acc, acc_norm]",a5321b7d032d95270a23d34a6d0c92da64c1d6996c74c2...,5439e357c8edeb897df57f00b4f43a6bc9879d8a4fd496...,d4735e3a265e16eee03f59718b9b5d03019c07d8b6c51f...,1.0,...,"[2, 1, 0, 3]",0.0,1.0,1,2,"[1, 0, 2, 3]","[2, 1, 0, 3]",True,False,False
4,4,3,"[[[-24.063854217529297, False]], [[-29.4297676...","[[-24.063854217529297, False], [-29.4297676086...",none,"[acc, acc_norm]",8959cd27ebc59ce3f66e3f0e896386066563fc89da2793...,6d4ee26ac87d2f203ffcb8f0497d12c9afac909a973a8e...,4e07408562bedb8b60ce05c1decfe3ad16b72230967de0...,0.0,...,"[2, 3, 0, 1]",0.0,0.0,2,2,"[2, 3, 0, 1]","[2, 3, 0, 1]",False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3031,3031,3,"[[[-31.122352600097656, False]], [[-17.3408241...","[[-31.122352600097656, False], [-17.3408241271...",none,"[acc, acc_norm]",0ce58ce2a73fa6ec512a9dec64f3c9dc58c6d4f7323d61...,723ee62df289ce158a0faa4830844ce2dc375f38c1de36...,4e07408562bedb8b60ce05c1decfe3ad16b72230967de0...,0.0,...,"[1, 2, 3, 0]",0.0,0.0,2,1,"[2, 0, 3, 1]","[1, 2, 3, 0]",False,False,False
3032,3032,0,"[[[-18.94678497314453, False]], [[-40.43418884...","[[-18.94678497314453, False], [-40.43418884277...",none,"[acc, acc_norm]",f40958c221ca69b19026a2a2ce053da57c57f08254b152...,12884805bd68bf3cf91f42be7784dd80661d3d2b86e872...,5feceb66ffc86f38d952786c6d696c79c2dbc239dd4e91...,1.0,...,"[0, 2, 3, 1]",1.0,1.0,0,0,"[0, 2, 3, 1]","[0, 2, 3, 1]",False,False,False
3033,3033,2,"[[[-24.67926788330078, False]], [[-20.71442794...","[[-24.67926788330078, False], [-20.71442794799...",none,"[acc, acc_norm]",dbeef71050b64e0a85ef4d362b36608c10939292ebb544...,27adbbb5a0b9f3aa53f1ed6e0f402556642493dd5b1d55...,d4735e3a265e16eee03f59718b9b5d03019c07d8b6c51f...,1.0,...,"[2, 1, 0, 3]",0.0,1.0,1,2,"[1, 2, 3, 0]","[2, 1, 0, 3]",False,False,False
3034,3034,3,"[[[-22.328153610229492, False]], [[-43.6439247...","[[-22.328153610229492, False], [-43.6439247131...",none,"[acc, acc_norm]",0c380fb17fba9c6037249a0e4a8888f77f34687a5c5ef3...,b7feecd8efff107c2c4294d00f55a9077a6bf90fceb8e9...,4e07408562bedb8b60ce05c1decfe3ad16b72230967de0...,0.0,...,"[0, 3, 1, 2]",0.0,0.0,0,0,"[0, 3, 1, 2]","[0, 3, 1, 2]",False,False,False


In [5]:

# Check that the model choices are consistent with accuracy, the label is called 'target'
assert all(
    (df_lm_eval_final_result['model_choice_init'] == df_lm_eval_final_result['target']) == (df_lm_eval_final_result['acc_init'] == 1)
), "Inconsistency found between model choices and accuracy in initial results."
assert all(
    (df_lm_eval_final_result['model_choice_final'] == df_lm_eval_final_result['target']) == (df_lm_eval_final_result['acc_final'] == 1)
), "Inconsistency found between model choices and accuracy in final results."

basic_performance_analysis(df_lm_eval_init_result)
basic_performance_analysis(df_lm_eval_final_result)

basic_performance_analysis(df_lm_eval_final_result, verdict_col_name="acc_final")
basic_performance_analysis(df_lm_eval_final_result, verdict_col_name="acc_init")

Overall Accuracy: 31.42% (954.0/3036)
Overall Accuracy: 44.66% (1356.0/3036)
Overall Accuracy: 44.66% (1356.0/3036)
Overall Accuracy: 31.42% (954.0/3036)


In [6]:
REPO_ID = "tomg-group-umd/fictionalqa_training_splits"
# LMEVAL_CONFIG_NAME = "fict_qa_obqa_blind_inf_ex_dedup_ds_mcq_topk4" alias
LMEVAL_CONFIG_NAME = "fict_qa_obqa_blind_inf_ex_dedup_ds_Llama-3-2-3B-Instruct_scored_rowlimNone_altlimNone_topk4_seed1234_slim"

df_hf_result = pull_raw_dataset(REPO_ID, LMEVAL_CONFIG_NAME)
# df_hf_result

In [7]:
# now join the two dataframes on index, they are the same order
df_merged_final = df_lm_eval_final_result.join(df_hf_result, lsuffix='_lm_eval', rsuffix='_hf')
# df_merged_final

In [8]:
# Add in the fictsheets and docs themselves

REPO_ID = "tomg-group-umd/fictionalqa"
CONFIG_NAME = "fictions"
df_fictions = pull_raw_dataset(REPO_ID, CONFIG_NAME)[["fiction_id","fiction"]]
CONFIG_NAME = "fictsheets"
df_fictsheets = pull_raw_dataset(REPO_ID, CONFIG_NAME)[["event_id","fictsheet"]]
CONFIG_NAME = "joined_qa"
df_joined_qa = pull_raw_dataset(REPO_ID, CONFIG_NAME)[["question_id","grade_informed","natural_answer_in_fiction"]]

df_merged_final = df_merged_final.merge(df_fictions, on="fiction_id", how="left").merge(
    df_fictsheets, on="event_id", how="left"
).merge(
    df_joined_qa, on="question_id", how="left"
)
# df_merged_final

In [9]:
# hand analysis columns

# for col in df_merged_final.columns:
#     print(f"\"{col}\",")

# cols_to_keep = [
#     # "doc_id",
#     # "target_lm_eval",
#     # "resps",
#     # "filtered_resps",
#     # "filter",
#     # "metrics",
#     # "doc_hash",
#     # "prompt_hash",
#     # "target_hash",
#     "acc",
#     # "acc_norm",
#     # "doc_event_id",
#     # "doc_fiction_id",
#     # "doc_question_id",
#     # "doc_span_answer",
#     # "doc_natural_answer",
#     # "doc_input",
#     # "doc_target",
#     # "doc_target_span",
#     # "doc_target_idx",
#     # "doc_topk_choices",
#     # "arguments_gen_args_0_arg_0",
#     # "arguments_gen_args_0_arg_1",
#     # "arguments_gen_args_1_arg_0",
#     # "arguments_gen_args_1_arg_1",
#     # "arguments_gen_args_2_arg_0",
#     # "arguments_gen_args_2_arg_1",
#     # "arguments_gen_args_3_arg_0",
#     # "arguments_gen_args_3_arg_1",
#     "event_id",
#     "fiction_id",
#     "question_id",
#     # "span_answer",
#     # "natural_answer",
#     "input",
#     "target_hf",
#     # "target_span",
#     "target_idx",
#     "topk_choices",
# ]
cols_to_keep = [
    "event_id",
    "fiction_id",
    "question_id",
    "fictsheet",
    "fiction",
    "input",
    "target_hf",
    "target_idx",
    "topk_choices",
    "choice_ranking_init",
    "choice_ranking_final",
    "W_and_E_in_choices",
    "W_and_E_is_model_choice_init",
    "W_and_E_is_model_choice_final",
    "model_choice_init",
    "model_choice_final",
    "acc_init",
    "acc_final",
    "grade_informed",
    "natural_answer_in_fiction",
]
df_merged_final = df_merged_final[cols_to_keep]

In [10]:
REPO_ID = "tomg-group-umd/fictionalqa_training_splits"
TRAIN_CONFIG_NAME = "event_split_fictions_webtext_train_ds_valratio0.33_seed1234_mcq_topk4"
VAL_CONFIG_NAME = "event_split_fictions_webtext_val_ds_valratio0.33_seed1234_mcq_topk4"

In [11]:
df_train = pull_raw_dataset(REPO_ID, TRAIN_CONFIG_NAME)
df_val = pull_raw_dataset(REPO_ID, VAL_CONFIG_NAME)
# df_train
# df_val


In [12]:
# basic_performance_analysis(df_lm_eval_final_result)
# basic_performance_analysis(df_merged_final)

In [13]:
df_merged_final_train = df_merged_final[df_merged_final['fiction_id'].isin(df_train['fiction_id'])]
print("Train Set Performance:")
basic_performance_analysis(df_merged_final_train, verdict_col_name="acc_init")
basic_performance_analysis(df_merged_final_train, verdict_col_name="acc_final")

Train Set Performance:
Overall Accuracy: 31.91% (633.0/1984)
Overall Accuracy: 45.31% (899.0/1984)


In [14]:
df_merged_final_val = df_merged_final[df_merged_final['fiction_id'].isin(df_val['fiction_id'])]
print("Validation Set Performance:")
basic_performance_analysis(df_merged_final_val, verdict_col_name="acc_init")
basic_performance_analysis(df_merged_final_val, verdict_col_name="acc_final")

Validation Set Performance:
Overall Accuracy: 30.51% (321.0/1052)
Overall Accuracy: 43.44% (457.0/1052)


In [15]:
# df_merged_final

In [16]:
# df_merged_final_train

In [17]:
# df_merged_final_val

In [18]:
# compute some confusion matrix like stats
# df_for_stats = df_merged_final
# df_for_stats = df_merged_final_train
df_for_stats = df_merged_final_val

# df_for_stats = df_for_stats.drop_duplicates(subset=['target_hf'], keep='first')
# df_for_stats = df_for_stats.drop_duplicates(subset=['target_hf'], keep='last')

# eg. fraction of rows whose acc_init was 1 that also have acc_final 1
num_init_correct = df_for_stats[df_for_stats['acc_init'] == 1].shape[0]
num_init_and_final_correct = df_for_stats[(df_for_stats['acc_init'] == 1) & (df_for_stats['acc_final'] == 1)].shape[0]
fraction_init_correct_final_correct = num_init_and_final_correct / num_init_correct if num_init_correct > 0 else 0
print(f"Fraction of initially correct answers that still remained correct after training: {fraction_init_correct_final_correct:.2%} ({num_init_and_final_correct}/{num_init_correct})")

# or fraction of final correct answers that were novel, i.e. not in initially correct
num_final_correct = df_for_stats[df_for_stats['acc_final'] == 1].shape[0]
num_final_novel_correct = df_for_stats[(df_for_stats['acc_init'] == 0) & (df_for_stats['acc_final'] == 1)].shape[0]
fraction_final_novel_correct = num_final_novel_correct / num_final_correct if num_final_correct > 0 else 0
print(f"Fraction of final correct answers that were learned i.e. not initially correct: {fraction_final_novel_correct:.2%} ({num_final_novel_correct}/{num_final_correct})")

# fraction that were final correct and novel, and also had a grade_informed of 1
num_final_novel_correct_with_grade_informed = df_for_stats[(df_for_stats['acc_init'] == 0) & (df_for_stats['acc_final'] == 1) & (df_for_stats['grade_informed'] == 1)].shape[0]
fraction_final_novel_correct_with_grade_informed = num_final_novel_correct_with_grade_informed / num_final_novel_correct if num_final_novel_correct > 0 else 0
print(f"Fraction of learned questions that also had grade_informed=1: {fraction_final_novel_correct_with_grade_informed:.2%} ({num_final_novel_correct_with_grade_informed}/{num_final_novel_correct})")

# fraction of questions that could be learned, i.e. were initially incorrect, and then became correct
num_learnable = df_for_stats[(df_for_stats['acc_init'] == 0)].shape[0]
num_learned = df_for_stats[(df_for_stats['acc_init'] == 0) & (df_for_stats['acc_final'] == 1)].shape[0]
fraction_learned = num_learned / num_learnable if num_learnable > 0 else 0
print(f"Fraction of learnable questions (initially incorrect) that were learned: {fraction_learned:.2%} ({num_learned}/{num_learnable})") 

# fraction of learned questions where "natural_answer_in_fiction" is 1
num_learned_with_natural_answer_in_fiction = df_for_stats[(df_for_stats['acc_init'] == 0) & (df_for_stats['acc_final'] == 1) & (df_for_stats['natural_answer_in_fiction'] == 1)].shape[0]
fraction_learned_with_natural_answer_in_fiction = num_learned_with_natural_answer_in_fiction / num_learned if num_learned > 0 else 0
print(f"Fraction of learned questions where natural_answer_in_fiction==1: {fraction_learned_with_natural_answer_in_fiction:.2%} ({num_learned_with_natural_answer_in_fiction}/{num_learned})")

# fraction of questions learnable but model failed to learn where "natural_answer_in_fiction" is 1
num_not_learned_with_natural_answer_in_fiction = df_for_stats[(df_for_stats['acc_init'] == 0) & (df_for_stats['acc_final'] == 0) & (df_for_stats['natural_answer_in_fiction'] == 1)].shape[0]
num_not_learned = df_for_stats[(df_for_stats['acc_init'] == 0) & (df_for_stats['acc_final'] == 0)].shape[0]
fraction_not_learned_with_natural_answer_in_fiction = num_not_learned_with_natural_answer_in_fiction / num_not_learned if num_not_learned > 0 else 0
print(f"Fraction of questions that model could have learned but did not where natural_answer_in_fiction==1: {fraction_not_learned_with_natural_answer_in_fiction:.2%} ({num_not_learned_with_natural_answer_in_fiction}/{num_not_learned})")

# other way around, for questions where natural_answer_in_fiction is 1, what fraction were learned vs possible to learn
num_learnable_with_natural_answer_in_fiction = df_for_stats[(df_for_stats['acc_init'] == 0) & (df_for_stats['natural_answer_in_fiction'] == 1)].shape[0]
fraction_learned_with_natural_answer_in_fiction = num_learned_with_natural_answer_in_fiction / num_learnable_with_natural_answer_in_fiction if num_learnable_with_natural_answer_in_fiction > 0 else 0
print(f"Fraction of learnable questions where natural_answer_in_fiction==1 that were actually learned: {fraction_learned_with_natural_answer_in_fiction:.2%} ({num_learned_with_natural_answer_in_fiction}/{num_learnable_with_natural_answer_in_fiction})")

# then for questions where natural_answer_in_fiction is 0, what fraction were learned vs possible to learn
num_learnable_with_natural_answer_not_in_fiction = df_for_stats[(df_for_stats['acc_init'] == 0) & (df_for_stats['natural_answer_in_fiction'] == 0)].shape[0]
num_learned_with_natural_answer_not_in_fiction = df_for_stats[(df_for_stats['acc_init'] == 0) & (df_for_stats['acc_final'] == 1) & (df_for_stats['natural_answer_in_fiction'] == 0)].shape[0]
fraction_learned_with_natural_answer_not_in_fiction = num_learned_with_natural_answer_not_in_fiction / num_learnable_with_natural_answer_not_in_fiction if num_learnable_with_natural_answer_not_in_fiction > 0 else 0
print(f"Fraction of learnable questions where natural_answer_in_fiction==0 that were actually learned: {fraction_learned_with_natural_answer_not_in_fiction:.2%} ({num_learned_with_natural_answer_not_in_fiction}/{num_learnable_with_natural_answer_not_in_fiction})")


# fraction with W and E as a choice
num_with_W_and_E_choice = df_for_stats[df_for_stats['W_and_E_in_choices'] == True].shape[0]
fraction_with_W_and_E_choice = num_with_W_and_E_choice / df_for_stats.shape[0] if df_for_stats.shape[0] > 0 else 0
print(f"Fraction of samples with 'W and E' as a choice: {fraction_with_W_and_E_choice:.2%} ({num_with_W_and_E_choice}/{df_for_stats.shape[0]})")

# or fraction of acquired final correct answers that had W and E as first choice initially, but then didnt and were 
num_final_correct_novel = df_for_stats[(df_for_stats['acc_init'] == 0) & (df_for_stats['acc_final'] == 1)].shape[0]
num_final_novel_W_and_E_demoted_correct = df_for_stats[(df_for_stats['acc_init'] == 0) & (df_for_stats['acc_final'] == 1) & (df_for_stats['W_and_E_is_model_choice_init'] == 1) & (df_for_stats['W_and_E_is_model_choice_final'] == 0)].shape[0]
fraction_final_novel_W_and_E_demoted_correct = num_final_novel_W_and_E_demoted_correct / num_final_correct_novel if num_final_correct_novel > 0 else 0
print(f"Fraction of learned questions that had 'W and E' as initial model choice but then demoted it: {fraction_final_novel_W_and_E_demoted_correct:.2%} ({num_final_novel_W_and_E_demoted_correct}/{num_final_correct_novel})")

# acquired correct questions after ignoring the W and E demotion cases, as a fraction questions that could be learned ignoring W and E demotion cases
num_final_novel_ignoring_W_and_E_demoted = num_final_novel_correct - num_final_novel_W_and_E_demoted_correct
num_learnable_ignoring_W_and_E_demoted = num_learnable - num_final_novel_W_and_E_demoted_correct
fraction_learned_ignoring_W_and_E_demoted = num_final_novel_ignoring_W_and_E_demoted / num_learnable_ignoring_W_and_E_demoted if num_learnable_ignoring_W_and_E_demoted > 0 else 0
print(f"Fraction of learnable questions that were learned ignoring W and E demotion cases: {fraction_learned_ignoring_W_and_E_demoted:.2%} ({num_final_novel_ignoring_W_and_E_demoted}/{num_learnable_ignoring_W_and_E_demoted})")



Fraction of initially correct answers that still remained correct after training: 85.36% (274/321)
Fraction of final correct answers that were learned i.e. not initially correct: 40.04% (183/457)
Fraction of learned questions that also had grade_informed=1: 89.62% (164/183)
Fraction of learnable questions (initially incorrect) that were learned: 25.03% (183/731)
Fraction of learned questions where natural_answer_in_fiction==1: 83.06% (152/183)
Fraction of questions that model could have learned but did not where natural_answer_in_fiction==1: 58.94% (323/548)
Fraction of learnable questions where natural_answer_in_fiction==1 that were actually learned: 32.00% (152/475)
Fraction of learnable questions where natural_answer_in_fiction==0 that were actually learned: 12.11% (31/256)
Fraction of samples with 'W and E' as a choice: 32.41% (341/1052)
Fraction of learned questions that had 'W and E' as initial model choice but then demoted it: 29.51% (54/183)
Fraction of learnable questions that

In [19]:
import spacy

# Load a larger spaCy model for better NER performance
# try:
#     nlp = spacy.load("en_core_web_sm")
# except OSError:
#     print("Downloading 'en_core_web_sm' model. This may take a moment.")
#     spacy.cli.download("en_core_web_sm")
#     nlp = spacy.load("en_core_web_sm")

nlp = spacy.load("en_core_web_sm")
# nlp = spacy.load("en_core_web_lg")

def calculate_distributional_overlap(
    col1: pd.Series,
    col2: pd.Series
) -> dict:
    """
    Measures the overlap between the entire contents of two string columns
    from a DataFrame using Jaccard Similarity on:
    1. Named Entities (using spaCy NER)
    2. All unique words (tokens).

    Args:
        col1: The first pandas Series (string column).
        col2: The second pandas Series (string column).

    Returns:
        A dictionary containing the calculated Jaccard Similarity measures.
    """

    # --- 1. Aggregate and Process Text ---

    # Concatenate all text in each column into a single large string
    # but shuffle before joining to avoid any order effects
    # col1 = col1.sample(frac=1, random_state=42).reset_index(drop=True)
    # col2 = col2.sample(frac=1, random_state=42).reset_index(drop=True)

    # text1_raw = " ".join(col1.astype(str))
    # text2_raw = " ".join(col2.astype(str))
    # using a newline sep to try and break up any accidental n-grams across rows
    # text1_raw = "\n".join(col1.astype(str))
    # text2_raw = "\n".join(col2.astype(str))

    # nah, best to create lists of ents per col and word by looping
    # --- 1.1 Process Text ---

    print("Processing text with spaCy, this may take a while...")
    # Process the aggregated text with spaCy
    # doc1 = nlp(text1_raw)
    # doc2 = nlp(text2_raw)
    docs_1 = list(nlp.pipe(col1.astype(str).tolist(), n_process=32))
    print("Text processing for col1 complete!")
    docs_2 = list(nlp.pipe(col2.astype(str).tolist(), n_process=32))
    print("Text processing for col2 complete!")

    # --- 2. Calculate Named Entity Jaccard Similarity ---

    # Extract all unique named entity strings (e.g., 'Apple Inc.', 'London')
    # from the entire content of each column.
    # entities1 = set([ent.text.lower() for ent in doc1.ents])
    # entities2 = set([ent.text.lower() for ent in doc2.ents])
    entities1 = set([ent.text.lower() for doc in docs_1 for ent in doc.ents])
    entities2 = set([ent.text.lower() for doc in docs_2 for ent in doc.ents])

    # Jaccard Similarity (Intersection / Union)
    # 
    entity_intersection = len(entities1.intersection(entities2))
    entity_union = len(entities1.union(entities2))

    # Avoid division by zero if both sets are empty
    entity_jaccard_similarity = (
        entity_intersection / entity_union if entity_union > 0 else 0.0
    )

    # --- 3. Calculate Word (Token) Jaccard Similarity ---

    # Extract all unique lowercased, non-punctuation, non-stopword tokens
    # (a basic form of distributional feature).
    # words1 = set([
    #     token.text.lower()
    #     for token in doc1
    #     if not token.is_punct and not token.is_space and not token.is_stop
    # ])
    # words2 = set([
    #     token.text.lower()
    #     for token in doc2
    #     if not token.is_punct and not token.is_space and not token.is_stop
    # ])
    words1 = set([
        token.text.lower()
        for doc in docs_1
        for token in doc
        if not token.is_punct and not token.is_space and not token.is_stop
    ])
    words2 = set([
        token.text.lower()
        for doc in docs_2
        for token in doc
        if not token.is_punct and not token.is_space and not token.is_stop
    ])

    word_intersection = len(words1.intersection(words2))
    word_union = len(words1.union(words2))

    word_jaccard_similarity = (
        word_intersection / word_union if word_union > 0 else 0.0
    )

    # --- 4. Conceptual ROUGE Note ---

    # For a ROUGE-like n-gram overlap measure:
    # ROUGE is typically used for sequence-to-sequence comparison (summary vs. reference).
    # To apply it here, you would treat `text1_raw` as the 'reference' and `text2_raw`
    # as the 'summary' and use a dedicated library like `rouge-score` or `py-rouge`.
    # A simple n-gram overlap is captured conceptually by the Word Jaccard (using unigrams).

    # --- 5. Return Results ---

    return {
        "entity_jaccard_similarity": entity_jaccard_similarity,
        "word_jaccard_similarity": word_jaccard_similarity,
        "entities_col1_count": len(entities1),
        "entities_col2_count": len(entities2),
        "words_col1_count": len(words1),
        "words_col2_count": len(words2),
        "entities_set_col1": entities1,
        "entities_set_col2": entities2,
        "words_set_col1": words1,
        "words_set_col2": words2,
    }

In [20]:
# there are two sets train and val, and two conditions init and final, which can either be 1 or 0, so there are 8 possible subsets
overlap_train_df = df_merged_final_train[(df_merged_final_train['acc_init'] == 0) & (df_merged_final_train['acc_final'] == 0)]
overlap_val_df = df_merged_final_val[(df_merged_final_val['acc_init'] == 0) & (df_merged_final_val['acc_final'] == 0)]
# # ...
# overlap_train_df = df_merged_final_train[(df_merged_final_train['acc_init'] == 1) & (df_merged_final_train['acc_final'] == 1)]
# overlap_val_df = df_merged_final_val[(df_merged_final_val['acc_init'] == 1) & (df_merged_final_val['acc_final'] == 1)]


calculate_distributional_overlap(
    overlap_train_df['target_hf'],
    overlap_val_df['target_hf']
)


Processing text with spaCy, this may take a while...


Text processing for col1 complete!
Text processing for col2 complete!


{'entity_jaccard_similarity': 0.017595307917888565,
 'word_jaccard_similarity': 0.10522496371552975,
 'entities_col1_count': 224,
 'entities_col2_count': 123,
 'words_col1_count': 936,
 'words_col2_count': 587,
 'entities_set_col1': {'1951',
  '1957',
  '1960s',
  '1961',
  '1962',
  '1986',
  '2004',
  '2021',
  '2047',
  '2054',
  '30-foot',
  '70',
  'acoustic',
  'acoustic technologies',
  'ai',
  'aiden cross',
  'amelia chen',
  'anabelle johnson',
  'angel of salisbury street',
  'annual',
  'april 2003',
  'arjun kohli',
  'armpit hair styling',
  'art collective',
  'arthur silvers',
  'ashwick anomaly research group',
  "beatrice clemens'",
  'beijing',
  'berlin',
  'britain',
  'buenos aires',
  'california',
  'cambridge',
  'cambridge collaborative writing festival',
  'carson whitmore',
  'carsonville',
  'carsonville forest',
  'carsonville local authorities',
  'carsonville research institute',
  'celtic',
  'centre pompidou',
  'coastal acoustic research institute',
 

In [ ]:
overlap_results = []

train_col_name = "fiction"
# val_col_name = "fiction"
# train_col_name = "target_hf"
# val_col_name = "target_hf"
# train_col_name = "input"
val_col_name = "input"

# dedupe_targets = True
dedupe_targets = False

for train_init_acc in [0, 1]:
    for train_final_acc in [0, 1]:
        for val_init_acc in [0, 1]:
            for val_final_acc in [0, 1]:
                print(f"Processing subset: Train Init Acc={train_init_acc}, Train Final Acc={train_final_acc}, Val Init Acc={val_init_acc}, Val Final Acc={val_final_acc}")

                overlap_train_df = df_merged_final_train[
                    (df_merged_final_train['acc_init'] == train_init_acc) &
                    (df_merged_final_train['acc_final'] == train_final_acc)
                ]
                overlap_val_df = df_merged_final_val[
                    (df_merged_final_val['acc_init'] == val_init_acc) &
                    (df_merged_final_val['acc_final'] == val_final_acc)
                ]

                if dedupe_targets:
                    overlap_train_df = overlap_train_df.drop_duplicates(subset=['target_hf'], keep='first')
                    overlap_val_df = overlap_val_df.drop_duplicates(subset=['target_hf'], keep='first')

                results = calculate_distributional_overlap(
                    overlap_train_df[train_col_name],
                    overlap_val_df[val_col_name]
                )

                # store just the similarity and count cols, so the numerical values with their key as rows in a dataframe
                row = {
                    "train_init_acc": train_init_acc,
                    "train_final_acc": train_final_acc,
                    "val_init_acc": val_init_acc,
                    "val_final_acc": val_final_acc,
                    "entity_jaccard_similarity": results["entity_jaccard_similarity"],
                    "word_jaccard_similarity": results["word_jaccard_similarity"],
                    "entities_col1_count": results["entities_col1_count"],
                    "entities_col2_count": results["entities_col2_count"],
                    "words_col1_count": results["words_col1_count"],
                    "words_col2_count": results["words_col2_count"],
                }
                overlap_results.append(row)

overlap_results_df = pd.DataFrame(overlap_results)
overlap_results_df

Processing subset: Train Init Acc=0, Train Final Acc=0, Val Init Acc=0, Val Final Acc=0
Processing text with spaCy, this may take a while...
Text processing for col1 complete!
Text processing for col2 complete!
Processing subset: Train Init Acc=0, Train Final Acc=0, Val Init Acc=0, Val Final Acc=1
Processing text with spaCy, this may take a while...
Text processing for col1 complete!
Text processing for col2 complete!
Processing subset: Train Init Acc=0, Train Final Acc=0, Val Init Acc=1, Val Final Acc=0
Processing text with spaCy, this may take a while...
Text processing for col1 complete!
Text processing for col2 complete!
Processing subset: Train Init Acc=0, Train Final Acc=0, Val Init Acc=1, Val Final Acc=1
Processing text with spaCy, this may take a while...
Text processing for col1 complete!
Text processing for col2 complete!
Processing subset: Train Init Acc=0, Train Final Acc=1, Val Init Acc=0, Val Final Acc=0
Processing text with spaCy, this may take a while...
Text processing

,train_init_acc,train_final_acc,val_init_acc,val_final_acc,entity_jaccard_similarity,word_jaccard_similarity,entities_col1_count,entities_col2_count,words_col1_count,words_col2_count
0,0,0,0,0,0.004369,0.031915,6724,173,20750,784
1,0,0,0,1,0.002062,0.015468,6724,80,20750,389
2,0,0,1,0,0.000593,0.005678,6724,30,20750,149
3,0,0,1,1,0.002635,0.021163,6724,125,20750,529
4,0,1,0,0,0.005991,0.043524,3689,173,14153,784
5,0,1,0,1,0.002927,0.020993,3689,80,14153,389
6,0,1,1,0,0.000807,0.007467,3689,30,14153,149
7,0,1,1,1,0.003684,0.028295,3689,125,14153,529
8,1,0,0,0,0.011192,0.068083,1363,173,7609,784
9,1,0,0,1,0.005575,0.033333,1363,80,7609,389


In [22]:
# Save the dataframe to a CSV file for further analysis
filename = f"distributional_overlap_results_deduptgt{dedupe_targets}_train-{train_col_name}_val-{val_col_name}_columns.csv"
filename

NameError: name 'dedupe_targets' is not defined

In [ ]:
# overlap_results_df.to_csv(filename, index=False)

In [32]:
# load a prev result
# filename = f"distributional_overlap_results_deduptgtTrue_train-fiction_val-fiction_columns.csv"
# filename = f"distributional_overlap_results_deduptgtTrue_train-target_hf_val-target_hf_columns.csv"
# filename = f"distributional_overlap_results_deduptgtTrue_train-fiction_val-target_hf_columns.csv"
# filename = f"distributional_overlap_results_deduptgtFalse_train-fiction_val-fiction_columns.csv"
# filename = f"distributional_overlap_results_deduptgtFalse_train-target_hf_val-target_hf_columns.csv"
filename = f"distributional_overlap_results_deduptgtFalse_train-fiction_val-target_hf_columns.csv"
# filename = f"distributional_overlap_results_deduptgtFalse_train-input_val-input_columns.csv"
# filename = f"distributional_overlap_results_deduptgtFalse_train-fiction_val-input_columns.csv"
overlap_results_df = pd.read_csv(f"analysis_dfs/{filename}")
overlap_results_df

,train_init_acc,train_final_acc,val_init_acc,val_final_acc,entity_jaccard_similarity,word_jaccard_similarity,entities_col1_count,entities_col2_count,words_col1_count,words_col2_count
0,0,0,0,0,0.002342,0.022230,6724,123,20750,587
1,0,0,0,1,0.000885,0.007066,6724,58,20750,201
2,0,0,1,0,0.000149,0.002986,6724,11,20750,78
3,0,0,1,1,0.005162,0.009657,6724,91,20750,264
4,0,1,0,0,0.003158,0.029114,3689,123,14153,587
5,0,1,0,1,0.001604,0.009494,3689,58,14153,201
6,0,1,1,0,0.000270,0.003809,3689,11,14153,78
7,0,1,1,1,0.008538,0.013070,3689,91,14153,264
8,1,0,0,0,0.006775,0.043545,1363,123,7609,587
9,1,0,0,1,0.002823,0.015341,1363,58,7609,201


In [33]:
markdown_cols = [
    "train_init_acc",
    "train_final_acc",
    "val_init_acc",
    "val_final_acc",
    "entity_jaccard_similarity",
    "word_jaccard_similarity",
    # "entities_col1_count",
    # "entities_col2_count",
    # "words_col1_count",
    # "words_col2_count",
]
md_table = overlap_results_df[markdown_cols].round(3).to_markdown()
print(md_table)

|    |   train_init_acc |   train_final_acc |   val_init_acc |   val_final_acc |   entity_jaccard_similarity |   word_jaccard_similarity |
|---:|-----------------:|------------------:|---------------:|----------------:|----------------------------:|--------------------------:|
|  0 |                0 |                 0 |              0 |               0 |                       0.002 |                     0.022 |
|  1 |                0 |                 0 |              0 |               1 |                       0.001 |                     0.007 |
|  2 |                0 |                 0 |              1 |               0 |                       0     |                     0.003 |
|  3 |                0 |                 0 |              1 |               1 |                       0.005 |                     0.01  |
|  4 |                0 |                 1 |              0 |               0 |                       0.003 |                     0.029 |
|  5 |                0 |  

In [40]:
# now lets pretty print some examples
# column_names are 
col_names = df_merged_final.columns.tolist()

cols_to_omit = [
    "fiction",
    "fictsheet",
    "model_choice_init",
    "model_choice_final",
    "W_and_E_in_choices",
    "W_and_E_is_model_choice_init",
    "W_and_E_is_model_choice_final",
    "grade_informed",
    "natural_answer_in_fiction",
]

num_samples = 20
seed = 1234
rng = np.random.default_rng(seed)

# df_to_print = df_merged_final
# df_to_print = df_merged_final_train
df_to_print = df_merged_final_val

# df_to_print = df_to_print.drop_duplicates(subset=['target_hf'])

# df_to_print = df_to_print[df_to_print['acc_init'] == 1]
# df_to_print = df_to_print[df_to_print['acc_final'] == 1]
# df_to_print = df_to_print[df_to_print['acc_init'] == 1 && df_to_print['acc_final'] == 1]
df_to_print = df_to_print[(df_to_print['acc_init'] == 0) & (df_to_print['acc_final'] == 1)]
# df_to_print = df_to_print[(df_to_print['acc_init'] == 0) & (df_to_print['acc_final'] == 0)]
# df_to_print = df_to_print[(df_to_print['acc_init'] == 1) & (df_to_print['acc_final'] == 0)]
# df_to_print = df_to_print[(df_to_print['acc_init'] == 0) & (df_to_print['acc_final'] == 0)]

samples = rng.choice(df_to_print.index, size=num_samples, replace=False)
# or just the first k
# samples = df_to_print.index[:num_samples]

for i in range(num_samples):
    idx = samples[i]
    print(f"=== Sample {i+1} (Index {idx}) ===")
    for col in filter(lambda c: c not in cols_to_omit,col_names):
        if col == "topk_choices":
            print(f"{col}:")
            for j, choice in enumerate(df_to_print.at[idx, col]):
                print(f"  Choice {j}: {choice}")
        else:
            print(f"{col}: {df_to_print.at[idx, col]}")
    print("\n")


=== Sample 1 (Index 2535) ===
event_id: event_084
fiction_id: event_084_style_corporate_num_000
question_id: event_084_style_corporate_num_000_question_004
input: Question: What did the diary become a symbol of?

Answer: 
target_hf: shared suffering
target_idx: 1
topk_choices:
  Choice 0: shared suffering and humanity
  Choice 1: shared suffering
  Choice 2: vulnerabilities
  Choice 3: vivid sketches and poignant accounts
choice_ranking_init: [2, 1, 0, 3]
choice_ranking_final: [1, 2, 0, 3]
acc_init: 0.0
acc_final: 1.0


=== Sample 2 (Index 1111) ===
event_id: event_036
fiction_id: event_036_style_news_num_002
question_id: event_036_style_news_num_002_question_001
input: Question: Who discovered the Waterfall Whisper phenomenon?

Answer: 
target_hf: Thomas Bright
target_idx: 3
topk_choices:
  Choice 0: tie
  Choice 1: Western and Eastern
  Choice 2: acoustic engineering and psychological principles
  Choice 3: Thomas Bright
choice_ranking_init: [0, 3, 1, 2]
choice_ranking_final: [3, 0, 